# Tesla and GameStop: Historical Stock Prices, Revenue, and Dashboards

Author: Vidhan Jain

Run all cells in an internet-connected Jupyter environment. Save the executed notebook before uploading to GitHub and taking screenshots. This file contains code, not pre-executed results.

In [ ]:
%pip install -q yfinance pandas requests beautifulsoup4 plotly

In [ ]:
import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
print("Libraries imported successfully.")

## Question 1: Use yfinance to Extract Tesla Stock Data

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
if tesla_data.empty:
    raise RuntimeError("No Tesla data returned. Retry this cell when Yahoo Finance is available.")
tesla_data.reset_index(inplace=True)
display(tesla_data.head())

## Question 2: Use Webscraping to Extract Tesla Revenue Data
Extract quarterly revenue from the course-hosted historical page. Values are in millions of US dollars.

In [ ]:
base_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/"
tesla_url = base_url + "revenue.htm"
response = requests.get(tesla_url, timeout=60)
response.raise_for_status()
html_data = response.text
soup = BeautifulSoup(html_data, "html.parser")
tesla_rows = []
for table in soup.find_all("table"):
    if "Tesla Quarterly Revenue" in table.get_text(" ", strip=True):
        for row in table.find_all("tr"):
            cols = row.find_all("td")
            if len(cols) >= 2:
                tesla_rows.append([cols[0].get_text(strip=True), cols[1].get_text(strip=True)])
        break
if not tesla_rows:
    raise ValueError("Tesla quarterly revenue table not found.")
tesla_revenue = pd.DataFrame(tesla_rows, columns=["Date", "Revenue"])
tesla_revenue["Revenue"] = tesla_revenue["Revenue"].str.replace(r"[$,]", "", regex=True).str.strip()
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue["Revenue"] != ""].copy()
display(tesla_revenue.tail())

## Question 3: Use yfinance to Extract GameStop Stock Data

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
if gme_data.empty:
    raise RuntimeError("No GameStop data returned. Retry this cell when Yahoo Finance is available.")
gme_data.reset_index(inplace=True)
display(gme_data.head())

## Question 4: Use Webscraping to Extract GameStop Revenue Data

In [ ]:
gme_url = base_url + "stock.html"
response = requests.get(gme_url, timeout=60)
response.raise_for_status()
html_data_2 = response.text
soup = BeautifulSoup(html_data_2, "html.parser")
gme_rows = []
for table in soup.find_all("table"):
    if "GameStop Quarterly Revenue" in table.get_text(" ", strip=True):
        for row in table.find_all("tr"):
            cols = row.find_all("td")
            if len(cols) >= 2:
                gme_rows.append([cols[0].get_text(strip=True), cols[1].get_text(strip=True)])
        break
if not gme_rows:
    raise ValueError("GameStop quarterly revenue table not found.")
gme_revenue = pd.DataFrame(gme_rows, columns=["Date", "Revenue"])
gme_revenue["Revenue"] = gme_revenue["Revenue"].str.replace(r"[$,]", "", regex=True).str.strip()
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue["Revenue"] != ""].copy()
display(gme_revenue.tail())

## Dashboard Function
Plot stock price and quarterly revenue through June 2021. Each dashboard has two panels. Current adjusted stock prices can differ from historical course screenshots.

In [ ]:
def make_graph(stock_data, revenue_data, stock_name):
    stock = stock_data.copy()
    revenue = revenue_data.copy()
    stock["Date"] = pd.to_datetime(stock["Date"], utc=True).dt.tz_convert(None).dt.normalize()
    revenue["Date"] = pd.to_datetime(revenue["Date"])
    revenue["Revenue"] = pd.to_numeric(revenue["Revenue"], errors="raise")
    cutoff = pd.Timestamp("2021-06-30")
    stock = stock[stock["Date"] <= cutoff].sort_values("Date")
    revenue = revenue[revenue["Date"] <= cutoff].sort_values("Date")
    if stock.empty or revenue.empty:
        raise ValueError("No data in the graph date range.")
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
        subplot_titles=(stock_name + " Historical Share Price", stock_name + " Historical Revenue"),
        vertical_spacing=0.15)
    fig.add_trace(go.Scatter(x=stock["Date"], y=stock["Close"], name="Share Price", line=dict(color="blue")), row=1, col=1)
    fig.add_trace(go.Scatter(x=revenue["Date"], y=revenue["Revenue"], name="Revenue", line=dict(color="green")), row=2, col=1)
    fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (USD millions)", row=2, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_layout(title=stock_name + " Historical Stock Price and Revenue", height=850, template="plotly_white", showlegend=False)
    fig.show()
print("make_graph is ready.")

## Question 5: Plot Tesla Stock and Revenue Dashboard

In [ ]:
make_graph(tesla_data, tesla_revenue, "Tesla")

## Question 6: Plot GameStop Stock and Revenue Dashboard

In [ ]:
make_graph(gme_data, gme_revenue, "GameStop")

## Question 7: Share the Assignment Notebook
Save this notebook after running all cells. Upload the executed .ipynb file to your GitHub repository, and paste its actual GitHub page URL in the assignment submission.

## Screenshot Checklist
1. Tesla extraction code and first five stock rows.
2. Tesla revenue last five rows.
3. GameStop extraction code and first five stock rows.
4. GameStop revenue last five rows.
5. Tesla dashboard, both panels.
6. GameStop dashboard, both panels.

## Data Sources
Stock data: Yahoo Finance through yfinance (TSLA and GME). Revenue: course-hosted historical Macrotrends snapshots linked in Questions 2 and 4.